In [1]:
# Recharge automatiquement src/transformers.py quand il est modifié
%load_ext autoreload
%autoreload 2

# Rend le dossier src/ (un niveau au-dessus de notebooks/) importable
import sys
sys.path.append("..")
from src.transformers import GroupMedianImputer, ModeImputer

import pandas as pd
import numpy as np
from src.preprocessing import clean, remove_outliers, add_features

from sklearn.pipeline import Pipeline


In [2]:

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
test_ids = test["Id"]  

train.shape, test.shape

((1460, 81), (1459, 80))

In [3]:
train = remove_outliers(train)

y = np.log1p(train["SalePrice"])

# Features uniquement : on retire l'identifiant et la cible
X_train = train.drop(columns=["Id", "SalePrice"])
X_test  = test.drop(columns=["Id"])

# Concaténation train + test ; n_train sert à re-séparer plus tard
n_train = len(X_train)
all_X = pd.concat([X_train, X_test], axis=0, ignore_index=True)
all_X.shape 

(2917, 79)

In [4]:
cleaned = clean(all_X)
cleaned_featured = add_features(cleaned)

In [5]:
X_train = cleaned_featured.iloc[:n_train].copy()
X_test = cleaned_featured.iloc[n_train:].copy()

In [6]:
print(X_train.shape) 
print(X_test.shape) 
print(y.shape) 

(1458, 80)
(1459, 80)
(1458,)


In [7]:
na = X_train.isna().sum()
print(na[na > 0])

LotFrontage     259
MasVnrType        5
BsmtExposure      1
BsmtFinType2      1
Electrical        1
dtype: int64


In [8]:
# Les 17 imputers (non entraînés) : ils seront fit dans le Pipeline, sur le train uniquement
imp_lotfrontage    = GroupMedianImputer()                                         # médiane par Neighborhood
mode_masvnrtype    = ModeImputer()                                                # mode hors "None"

imp_garage_area    = GroupMedianImputer(target_col="GarageArea", group_col="GarageType")
imp_garage_cars    = GroupMedianImputer(target_col="GarageCars", group_col="GarageType")
mode_garage_finish = ModeImputer(target_col="GarageFinish")
mode_garage_qual   = ModeImputer(target_col="GarageQual")
mode_garage_cond   = ModeImputer(target_col="GarageCond")

mode_bsmt_cond     = ModeImputer(target_col="BsmtCond")
mode_bsmt_qual     = ModeImputer(target_col="BsmtQual")
mode_bsmt_exposure = ModeImputer(target_col="BsmtExposure")
mode_bsmt_fintype2 = ModeImputer(target_col="BsmtFinType2", exclude_value=["None", "Unf"])  # type aménagé le plus fréquent

mode_mszoning      = ModeImputer(target_col="MSZoning")
mode_exterior1st   = ModeImputer(target_col="Exterior1st")
mode_exterior2nd   = ModeImputer(target_col="Exterior2nd")
mode_electrical    = ModeImputer(target_col="Electrical")
mode_kitchenqual   = ModeImputer(target_col="KitchenQual")
mode_saletype      = ModeImputer(target_col="SaleType")

In [9]:
# Pipeline des 17 imputers : fit sur le train uniquement, puis appliqué au train et au test
pipe = Pipeline([
    ("lotfrontage",    imp_lotfrontage),
    ("masvnrtype",     mode_masvnrtype),
    ("garage_area",    imp_garage_area),
    ("garage_cars",    imp_garage_cars),
    ("garage_finish",  mode_garage_finish),
    ("garage_qual",    mode_garage_qual),
    ("garage_cond",    mode_garage_cond),
    ("bsmt_cond",      mode_bsmt_cond),
    ("bsmt_qual",      mode_bsmt_qual),
    ("bsmt_exposure",  mode_bsmt_exposure),
    ("bsmt_fintype2",  mode_bsmt_fintype2),
    ("mszoning",       mode_mszoning),
    ("exterior1st",    mode_exterior1st),
    ("exterior2nd",    mode_exterior2nd),
    ("electrical",     mode_electrical),
    ("kitchenqual",    mode_kitchenqual),
    ("saletype",       mode_saletype),
])

pipe.fit(X_train)
X_train_imp = pipe.transform(X_train)
X_test_imp  = pipe.transform(X_test)

In [10]:
na_train_imp = X_train_imp.isna().sum()
print(na_train_imp[na_train_imp > 0])

Series([], dtype: int64)


In [11]:
na_test_imp = X_test_imp.isna().sum()
print(na_test_imp[na_test_imp > 0])

Series([], dtype: int64)


In [12]:
X_train_imp.shape, X_test_imp.shape

((1458, 80), (1459, 80))

In [13]:
type(X_train_imp)  

pandas.DataFrame

In [14]:
type(X_test_imp)  

pandas.DataFrame

In [16]:
pipe.named_steps["bsmt_fintype2"].mode_

'Rec'